In [19]:
!pip install huggingface_hub
!pip install 'huggingface_hub[cli]'

In [1]:
import os
import cv2
from PIL import Image
from datasets import Dataset
import itertools

In [2]:
input_folder = '../data/preprocessed/upscaled'

In [18]:
target_size = 1024

def resize(image):
    """Resize image maintaining aspect ratio with longest side as target_size."""
    height, width = image.shape[:2]
    scale_ratio = target_size / max(height, width)
    new_width = int(width * scale_ratio)
    new_height = target_size
    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)

In [4]:
image_groups = {}
for root, _, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.png'):
            try:
                id_part, number = file.rsplit('_', 1)
                number = number[:-4]  # Remove .png
                file_path = os.path.join(root, file)
                
                if id_part not in image_groups:
                    image_groups[id_part] = []
                image_groups[id_part].append((number, file_path))
            except ValueError:
                continue

In [20]:
dataset_data = []
for id_key, files in image_groups.items():
    # Sort files by number
    files.sort(key=lambda x: x[0])
    numbers, file_paths = zip(*files)
    
    # Generate all unique pairs
    for (i, j) in itertools.combinations(range(len(file_paths)), 2):
        img1_path = file_paths[i]
        img2_path = file_paths[j]
        
        # Process first image
        img1 = cv2.imread(img1_path)
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        img1 = resize(img1)
        
        # Process second image
        img2 = cv2.imread(img2_path)
        img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
        img2 = resize(img2)
        
        # Create dataset entry
        dataset_data.append({
            'image1': Image.fromarray(img1),
            'image2': Image.fromarray(img2),
            'name1': f"{id_key}_{numbers[i]}",
            'name2': f"{id_key}_{numbers[j]}"
        })


In [21]:
hf_dataset = Dataset.from_list(dataset_data)


In [22]:
hf_dataset

Dataset({
    features: ['image1', 'image2', 'name1', 'name2'],
    num_rows: 2486
})

In [23]:
from huggingface_hub import notebook_login
notebook_login()

In [25]:
hf_dataset.push_to_hub("Zaurall/floor_coverings_1")


Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map:   0%|          | 0/829 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Map:   0%|          | 0/829 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Zaurall/floor_coverings_1/commit/c68f3d8484aa9ae9af02fbe4a22913874c23a1c4', commit_message='Upload dataset', commit_description='', oid='c68f3d8484aa9ae9af02fbe4a22913874c23a1c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Zaurall/floor_coverings_1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Zaurall/floor_coverings_1'), pr_revision=None, pr_num=None)